# 03 — Run the Trained Model Live

Load the MOUSE model pushed by `02_train_offline.ipynb` and run it in a live FrozenLake environment.

At each step the model receives the most recent `(action, observation, reward, done)` and picks the action with the highest predicted Q-value. The episode is captured frame-by-frame and played back as an inline animation.

In [9]:
import os

import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from mouse_envs import EnvConfig, make_env
from mouse_core import load_model

# FrozenLake renders via pygame; run headless in environments without a display.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

# Hub model repo written by 02_train_offline.ipynb.
MODEL_ID = "mouse-example-model"

# FrozenLake has four discrete actions, matching the DQN head in the training notebook.
MAX_ACTIONS = 4

# Number of steps to run the model in the live environment.
EVAL_STEPS = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Environment

We use the slippery FrozenLake variant with `render_mode="rgb_array"` so we can capture frames for the replay animation.

In [10]:
env = make_env(EnvConfig(
    "FrozenLake-v1",
    name="frozenlake_slippery",
    seed=0,
    num_envs=1,
    episodes_per_task=5,
    kwargs={"is_slippery": True, "render_mode": "rgb_array"},
))

## Load model

Download the checkpoint from the Hub if needed, reconstruct the saved MOUSE model, and move it to the selected device.

In [11]:
model = load_model(MODEL_ID, map_location="cpu").eval().to(device)

## Incremental inference with KV-cache

At inference time you don't need to re-process the full history on every step. Backbones that support it (`LlamaBackbone`, `Qwen3Backbone`) cache the key/value states from previous tokens so each new step only processes the single new token.

Pass `use_cache=True` and thread the cache object across steps:

```python
cache = None

with torch.no_grad():
    for step in range(EVAL_STEPS):
        predictions, _, cache = model(batch, cache=cache, use_cache=True)
        action = model.get_action(out, temperature=0.0, num_actions=MAX_ACTIONS)
```

`model.get_action` reads the configured `action_head` from the last step of `out`. Use `temperature=0.0` for greedy argmax and `temperature > 0` for sampling. Pass `num_actions` when the live environment has fewer valid actions than the head maximum.

## Run an episode

`model_step` takes the action we just sent and the observation that came back, feeds them to the model, and returns the next action to take. We kick off the loop with a single random action so there's something to feed the model on the first call.

In [ ]:
def model_step(inputs_per_env, outputs, cache):
    """Feed the last transition to the model and return the next inputs + updated cache."""
    obs = outputs[0]
    batch = [[{
        "action": inputs_per_env[0]["action"],
        "observation": obs["observation"],
        "reward": obs["reward"],
        "done": obs["done"],
    }]]
    predictions, _, cache = model(batch, cache=cache, use_cache=True)
    action = model.get_action(out, temperature=0.0, num_actions=MAX_ACTIONS)
    return [{"action": action.squeeze().cpu()}], cache


inputs_per_env = env.sample_random_inputs()
cache = None
frames = []

with torch.no_grad():
    for _ in range(EVAL_STEPS):
        outputs = env.step(inputs_per_env)
        frames.extend(env.render())
        inputs_per_env, cache = model_step(inputs_per_env, outputs, cache)

env.close()
print(f"Collected {len(frames)} frames")

Collected 50 frames


## Replay

Watch the episode play back. Each frame is one environment step.

In [13]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.axis("off")
img = ax.imshow(frames[0])

def update(frame):
    img.set_data(frame)
    return (img,)

ani = animation.FuncAnimation(fig, update, frames=frames, interval=200, blit=True)
plt.close(fig)

HTML(ani.to_jshtml())